# LSTM Autoencoder (strict normal-only)

Sequence-to-sequence LSTM encoder-decoder, reconstructing the 30-step window. Trained STRICTLY on normal-only data (unlike the broader-scope models), so it answers a narrower question: does this look like normal RUNNING, specifically — not just 'a benign state in general'.

Trained STRICTLY on `normal` sessions only — motor_off and transition sessions are excluded entirely, so this model's job is simply: does this look like normal running, or not.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn tensorflow joblib openpyxl matplotlib

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    RepeatVector,
    TimeDistributed,
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
)


In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Change only this path.
BASE_PATH = (
    r"../../data"
    r"/Motor_data_model"
)

# Raw data values are stored as scaled integers.
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]

N_FEATURES = len(FEATURE_COLUMNS)

WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
EPOCHS = 100
BATCH_SIZE = 64

RECON_THRESHOLD_PERCENTILE = 99.0
FEATURE_THRESHOLD_PERCENTILE = 99.0

# Available modes:
# "mean"   = reconstruction MSE only
# "max"    = maximum feature-ratio only
# "hybrid" = either condition can trigger anomaly
SCORE_MODE = "hybrid"

CLIP_SCALED_INPUT = False

FILE_ANOMALY_FRACTION_THRESHOLD = 0.10

OUTPUT_DIR = "pure_normal_lstm_ae_output"

CV_DIR = os.path.join(
    OUTPUT_DIR,
    "cross_validation",
)

FINAL_DIR = os.path.join(
    OUTPUT_DIR,
    "final_model",
)

FINAL_EVAL_DIR = os.path.join(
    OUTPUT_DIR,
    "final_model_evaluation",
)

MANUAL_TEST_DIR = os.path.join(
    OUTPUT_DIR,
    "manual_test_results",
)

for folder in [
    CV_DIR,
    FINAL_DIR,
    FINAL_EVAL_DIR,
    MANUAL_TEST_DIR,
]:
    os.makedirs(
        folder,
        exist_ok=True,
    )

print("TensorFlow:", tf.__version__)
print("Features:", N_FEATURES)
print("Input shape:", (WINDOW_LEN, N_FEATURES))


In [ ]:
# ==========================================================
# CELL 3 — normal and anomaly file detection
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def normal_state_for(path):
    p = normalized_path(path)
    stem = Path(p).stem.lower()

    excluded_words = [
        "motor_off",
        "motor_being_on",
        "motor_being_off",
        "transition",
        "off_on_spikes",
    ]

    if any(word in p for word in excluded_words):
        return None

    if (
        "/normal/" in p
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal_running"

    return None


def anomaly_label_for(path):
    p = normalized_path(path)

    if "motor_being_on" in p:
        return "motor_being_on"

    if "motor_being_off" in p:
        return "motor_being_off"

    if (
        "/motor_off/" in p
        or "motor_off" in Path(p).stem.lower()
    ):
        return "motor_off"

    if (
        "electrical fault" in p
        or "abnormal_current" in p
    ):
        return "electrical_fault"

    if (
        "mechanical fault" in p
        or "abnormal_vibration" in p
    ):
        return "mechanical_fault"

    if (
        "thermal fault" in p
        or "abnormal_temp" in p
    ):
        return "thermal_fault"

    return None


In [ ]:
# ==========================================================
# CELL 4 — discover dataset files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(
            base_path,
            "**",
            "*.xlsx",
        ),
        recursive=True,
    )

    files += glob.glob(
        os.path.join(
            base_path,
            "**",
            "*.csv",
        ),
        recursive=True,
    )

    return sorted(set(files))


files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

print(
    "Total files discovered:",
    len(files),
)

In [ ]:
# ==========================================================
# CELL 5 — read one file
# ==========================================================
def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)

    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

    else:
        raise ValueError(
            f"Unsupported file type: {path}"
        )

    missing = [
        column
        for column in FEATURE_COLUMNS
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns in {path.name}: {missing}"
        )

    clean = df[FEATURE_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce",
    )

    clean = clean / SCALE_DIVISOR

    clean = clean.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only {len(clean)} valid rows. "
            f"At least {WINDOW_LEN} rows are required."
        )

    return clean
    


In [ ]:
# ==========================================================
# CELL 6 — load only normal WITH-LOAD data for training
# ==========================================================
def is_normal_with_load(path):
    p = (
        normalized_path(path)
        .replace(" ", "_")
        .replace("-", "_")
    )

    # Reject normal files recorded without load.
    without_load_markers = [
        "without_load",
        "withoutload",
        "no_load",
        "noload",
        "unloaded",
    ]

    if any(marker in p for marker in without_load_markers):
        return False

    # Accept normal files recorded with load.
    with_load_markers = [
        "with_load",
        "withload",
        "normal_load",
        "normal_loaded",
    ]

    return any(marker in p for marker in with_load_markers)


healthy_sessions = []
fault_sessions = []
excluded_normal_files = []

for file_path in files:
    normal_state = normal_state_for(file_path)
    fault_label = anomaly_label_for(file_path)

    if normal_state is None and fault_label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    # Only normal WITH-LOAD files can enter healthy_sessions.
    if (
        normal_state is not None
        and not is_normal_with_load(file_path)
    ):
        relative_path = os.path.relpath(
            file_path,
            BASE_PATH,
        )

        excluded_normal_files.append(relative_path)

        print(
            "EXCLUDED normal without load:",
            relative_path,
        )
        continue

    try:
        df = read_one(file_path)

    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    record = {
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(
            file_path,
            BASE_PATH,
        ),
        "rows": len(df),
    }

    if normal_state is not None:
        record.update({
            "state": normal_state,
            "load_condition": "with_load",
            "binary_label": 0,
        })

        healthy_sessions.append(record)

        print(
            f"N | {'normal_with_load':20s} | "
            f"{len(df):6d} | "
            f"{record['file']}"
        )

    else:
        record.update({
            "fault_label": fault_label,
            "binary_label": 1,
        })

        fault_sessions.append(record)

        print(
            f"A | {fault_label:20s} | "
            f"{len(df):6d} | "
            f"{record['file']}"
        )


if not healthy_sessions:
    raise RuntimeError(
        "No normal WITH-LOAD files were loaded. "
        "Check the file or folder names."
    )

if len(healthy_sessions) < N_SPLITS:
    raise RuntimeError(
        f"Only {len(healthy_sessions)} normal with-load files "
        f"were loaded, but N_SPLITS={N_SPLITS}."
    )

if not fault_sessions:
    raise RuntimeError(
        "No anomaly files were loaded."
    )


print("\nNORMAL WITH-LOAD FILES USED FOR TRAINING")

for session in healthy_sessions:
    print("  USED:", session["file"])


print("\nNORMAL WITHOUT-LOAD FILES EXCLUDED")

for file_name in excluded_normal_files:
    print("  EXCLUDED:", file_name)


print("\nDATASET SUMMARY")
print("Normal with-load files:", len(healthy_sessions))
print("Excluded normal files:", len(excluded_normal_files))
print("Fault evaluation files:", len(fault_sessions))

print("\nFAULT FILE DISTRIBUTION")

print(
    pd.Series([
        session["fault_label"]
        for session in fault_sessions
    ]).value_counts()
)

In [ ]:
# ==========================================================
# CELL 7 — window generation
# ==========================================================
def make_windows_from_array(
    X_raw,
    scaler,
):
    X_scaled = scaler.transform(
        X_raw
    ).astype(np.float32)

    if CLIP_SCALED_INPUT:
        X_scaled = np.clip(
            X_scaled,
            0.0,
            1.0,
        )

    starts = np.arange(
        0,
        len(X_scaled) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_scaled[
            start:start + WINDOW_LEN
        ]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_windows_from_sessions(
    sessions,
    scaler,
):
    all_windows = []

    for session in sessions:
        windows, _ = make_windows_from_array(
            session["X"],
            scaler,
        )

        if windows is not None:
            all_windows.append(
                windows
            )

    if not all_windows:
        return np.empty(
            (
                0,
                WINDOW_LEN,
                N_FEATURES,
            ),
            dtype=np.float32,
        )

    return np.concatenate(
        all_windows,
        axis=0,
    ).astype(np.float32)


In [ ]:
# ==========================================================
# CELL 8 — LSTM autoencoder model
# ==========================================================
def window_mse_loss(
    y_true,
    y_pred,
):
    return tf.reduce_mean(
        tf.square(
            y_true - y_pred
        ),
        axis=[1, 2],
    )


def build_autoencoder_model(
    window_len,
    n_features,
):
    inputs = Input(
        shape=(
            window_len,
            n_features,
        ),
        name="float_input",
    )

    # Encoder
    x = LSTM(
        units=8,
        return_sequences=True,
        unroll = True,
        activation="tanh",
        recurrent_activation="sigmoid",
        name="encoder_lstm",
    )(inputs)

    x = Dropout(
        rate=0.05,
        name="encoder_dropout",
    )(x)

    latent = LSTM(
        units=4,
        return_sequences=False,
        unroll = True,
        activation="tanh",
        recurrent_activation="sigmoid",
        name="latent",
    )(x)

    # Decoder
    x = RepeatVector(
        window_len,
        name="repeat_latent",
    )(latent)

    x = LSTM(
        units=8,
        return_sequences=True,
        unroll = True,
        activation="tanh",
        recurrent_activation="sigmoid",
        name="decoder_lstm",
    )(x)

    outputs = TimeDistributed(
        Dense(
            n_features,
            activation="linear",
        ),
        name="reconstruction_output",
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="pure_normal_lstm_autoencoder",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss=window_mse_loss,
    )

    return model


def make_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=15,
            restore_best_weights=True,
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=5,
            min_lr=1e-5,
        ),
    ]


temporary_model = build_autoencoder_model(
    WINDOW_LEN,
    N_FEATURES,
)

temporary_model.summary()

del temporary_model


In [ ]:
# ==========================================================
# CELL 9 — anomaly scoring and threshold calibration
# ==========================================================
def anomaly_score(
    model,
    X,
):
    reconstruction = model.predict(
        X,
        verbose=0,
    )

    per_feature_error = np.mean(
        np.square(
            X - reconstruction
        ),
        axis=1,
    ).astype(np.float32)

    reconstruction_mse = np.mean(
        per_feature_error,
        axis=1,
    ).astype(np.float32)

    worst_feature_ids = np.argmax(
        per_feature_error,
        axis=1,
    )

    return (
        reconstruction_mse,
        per_feature_error,
        worst_feature_ids,
    )


def calibrate_thresholds(
    model,
    X_normal,
):
    (
        reconstruction_mse,
        per_feature_error,
        _,
    ) = anomaly_score(
        model,
        X_normal,
    )

    reconstruction_threshold = float(
        np.percentile(
            reconstruction_mse,
            RECON_THRESHOLD_PERCENTILE,
        )
    )

    feature_thresholds = np.percentile(
        per_feature_error,
        FEATURE_THRESHOLD_PERCENTILE,
        axis=0,
    ).astype(np.float32)

    feature_ratios = (
        per_feature_error
        / np.maximum(
            feature_thresholds.reshape(
                1,
                -1,
            ),
            1e-12,
        )
    )

    maximum_feature_ratio = np.max(
        feature_ratios,
        axis=1,
    )

    max_ratio_threshold = float(
        np.percentile(
            maximum_feature_ratio,
            FEATURE_THRESHOLD_PERCENTILE,
        )
    )

    return (
        reconstruction_threshold,
        feature_thresholds,
        max_ratio_threshold,
    )


def decisions_from_errors(
    reconstruction_mse,
    per_feature_error,
    reconstruction_threshold,
    feature_thresholds,
    max_ratio_threshold,
):
    mean_flag = (
        reconstruction_mse
        > reconstruction_threshold
    )

    feature_ratios = (
        per_feature_error
        / np.maximum(
            feature_thresholds.reshape(
                1,
                -1,
            ),
            1e-12,
        )
    )

    maximum_feature_ratio = np.max(
        feature_ratios,
        axis=1,
    )

    feature_flag = (
        maximum_feature_ratio
        > max_ratio_threshold
    )

    if SCORE_MODE == "mean":
        flags = mean_flag

    elif SCORE_MODE == "max":
        flags = feature_flag

    elif SCORE_MODE == "hybrid":
        flags = (
            mean_flag
            | feature_flag
        )

    else:
        raise ValueError(
            "SCORE_MODE must be mean, max, or hybrid."
        )

    return (
        flags.astype(np.int64),
        maximum_feature_ratio.astype(
            np.float32
        ),
    )

In [ ]:
# ==========================================================
# CELL 10 — file-level and window-level predictions
# ==========================================================
def aggregate_predictions(
    sessions,
    model,
    scaler,
    reconstruction_threshold,
    feature_thresholds,
    max_ratio_threshold,
):
    file_rows = []
    window_rows = []

    for session in sessions:
        windows, starts = make_windows_from_array(
            session["X"],
            scaler,
        )

        if windows is None:
            continue

        (
            reconstruction_mse,
            per_feature_error,
            worst_feature_ids,
        ) = anomaly_score(
            model,
            windows,
        )

        (
            flags,
            maximum_feature_ratio,
        ) = decisions_from_errors(
            reconstruction_mse,
            per_feature_error,
            reconstruction_threshold,
            feature_thresholds,
            max_ratio_threshold,
        )

        true_label = int(
            session["binary_label"]
        )

        state = session.get(
            "state",
            session.get(
                "fault_label",
                "unknown",
            ),
        )

        anomaly_fraction = float(
            np.mean(flags)
        )

        file_prediction = int(
            anomaly_fraction
            >= FILE_ANOMALY_FRACTION_THRESHOLD
        )

        file_rows.append({
            "file": session["file"],
            "state": state,
            "true_label": true_label,
            "pred_label": file_prediction,
            "windows": len(windows),
            "anomaly_windows": int(
                np.sum(flags)
            ),
            "anomaly_fraction":
                anomaly_fraction,
            "mean_mse": float(
                np.mean(
                    reconstruction_mse
                )
            ),
            "max_mse": float(
                np.max(
                    reconstruction_mse
                )
            ),
            "reconstruction_threshold":
                reconstruction_threshold,
            "max_ratio_threshold":
                max_ratio_threshold,
        })

        for window_id in range(
            len(windows)
        ):
            window_rows.append({
                "file":
                    session["file"],

                "state":
                    state,

                "window":
                    window_id,

                "start_row":
                    int(
                        starts[window_id]
                    ),

                "true_label":
                    true_label,

                "pred_label":
                    int(
                        flags[window_id]
                    ),

                "reconstruction_mse":
                    float(
                        reconstruction_mse[
                            window_id
                        ]
                    ),

                "reconstruction_threshold":
                    reconstruction_threshold,

                "max_feature_ratio":
                    float(
                        maximum_feature_ratio[
                            window_id
                        ]
                    ),

                "max_ratio_threshold":
                    max_ratio_threshold,

                "worst_feature":
                    FEATURE_COLUMNS[
                        int(
                            worst_feature_ids[
                                window_id
                            ]
                        )
                    ],
            })

    return (
        pd.DataFrame(
            file_rows
        ),
        pd.DataFrame(
            window_rows
        ),
    )

In [ ]:
# ==========================================================
# CELL 11 — build cross-validation table
# ==========================================================
all_sessions = (
    healthy_sessions
    + fault_sessions
)

cv_table = pd.DataFrame({
    "index": np.arange(
        len(all_sessions)
    ),

    "label": [
        session["binary_label"]
        for session in all_sessions
    ],

    "group": [
        session["file"]
        for session in all_sessions
    ],
})

print(
    cv_table["label"].value_counts()
)

In [ ]:
# ==========================================================
# CELL 12 — five-fold session-wise cross-validation
# ==========================================================
splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
cv_file_results = []
cv_window_results = []
cv_test_files = []

for fold, (
    train_index,
    test_index,
) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print(
        "\n"
        + "=" * 70
    )

    print(
        f"CV FOLD {fold}/{N_SPLITS}"
    )

    print(
        "=" * 70
    )

    train_sessions = [
        all_sessions[index]
        for index in train_index
    ]

    test_sessions = [
        all_sessions[index]
        for index in test_index
    ]

    print("\nTEST FILES")

    for session in test_sessions:
        state = session.get(
            "state",
            session.get(
                "fault_label",
                "unknown",
            ),
        )

        cv_test_files.append({
            "fold": fold,
            "file": session["file"],
            "state": state,
            "true_label":
                session["binary_label"],
        })

        print(
            f"label={session['binary_label']} | "
            f"state={state} | "
            f"file={session['file']}"
        )

    normal_train_sessions = [
        session
        for session in train_sessions
        if session["binary_label"] == 0
    ]

    rng = np.random.default_rng(
        SEED + fold
    )

    shuffled_ids = rng.permutation(
        len(normal_train_sessions)
    )

    number_calibration_files = max(
        1,
        round(
            0.20
            * len(shuffled_ids)
        ),
    )

    calibration_ids = set(
        shuffled_ids[
            :number_calibration_files
        ].tolist()
    )

    fit_sessions = [
        session
        for index, session
        in enumerate(
            normal_train_sessions
        )
        if index not in calibration_ids
    ]

    calibration_sessions = [
        session
        for index, session
        in enumerate(
            normal_train_sessions
        )
        if index in calibration_ids
    ]

    print(
        "\nNormal fit files:",
        len(fit_sessions),
    )

    print(
        "Normal calibration files:",
        len(calibration_sessions),
    )

    scaler = MinMaxScaler(
        feature_range=(0, 1)
    )

    scaler.fit(
        np.concatenate([
            session["X"]
            for session in fit_sessions
        ])
    )

    X_fit = build_windows_from_sessions(
        fit_sessions,
        scaler,
    )

    X_calibration = (
        build_windows_from_sessions(
            calibration_sessions,
            scaler,
        )
    )

    tf.keras.backend.clear_session()

    model = build_autoencoder_model(
        WINDOW_LEN,
        N_FEATURES,
    )

    model.fit(
        X_fit,
        X_fit,

        validation_data=(
            X_calibration,
            X_calibration,
        ),

        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(),
        verbose=1,
        shuffle=True,
    )

    (
        reconstruction_threshold,
        feature_thresholds,
        max_ratio_threshold,
    ) = calibrate_thresholds(
        model,
        X_calibration,
    )

    (
        fold_file_df,
        fold_window_df,
    ) = aggregate_predictions(
        test_sessions,
        model,
        scaler,
        reconstruction_threshold,
        feature_thresholds,
        max_ratio_threshold,
    )

    fold_file_df.insert(
        0,
        "fold",
        fold,
    )

    fold_window_df.insert(
        0,
        "fold",
        fold,
    )

    cv_file_results.append(
        fold_file_df
    )

    cv_window_results.append(
        fold_window_df
    )

    y_true = fold_window_df[
        "true_label"
    ].to_numpy()

    y_pred = fold_window_df[
        "pred_label"
    ].to_numpy()

    y_score = fold_window_df[
        "reconstruction_mse"
    ].to_numpy()

    fold_result = {
        "fold": fold,

        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "roc_auc": (
            roc_auc_score(
                y_true,
                y_score,
            )
            if len(
                np.unique(y_true)
            ) == 2
            else np.nan
        ),

        "reconstruction_threshold":
            reconstruction_threshold,

        "max_ratio_threshold":
            max_ratio_threshold,
    }

    cv_metrics.append(
        fold_result
    )

    print(
        "\nWINDOW-LEVEL FOLD RESULTS"
    )

    print(
        pd.Series(
            fold_result
        )
    )

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "normal",
                "anomaly",
            ],
            zero_division=0,
        )
    )

In [ ]:
# ==========================================================
# CELL 13 — combined cross-validation results
# ==========================================================
cv_metrics_df = pd.DataFrame(
    cv_metrics
)

cv_test_files_df = pd.DataFrame(
    cv_test_files
)

cv_file_results_df = pd.concat(
    cv_file_results,
    ignore_index=True,
)

cv_window_results_df = pd.concat(
    cv_window_results,
    ignore_index=True,
)

cv_metrics_df.to_csv(
    os.path.join(
        CV_DIR,
        "cv_metrics.csv",
    ),
    index=False,
)

cv_test_files_df.to_csv(
    os.path.join(
        CV_DIR,
        "cv_test_files.csv",
    ),
    index=False,
)

cv_file_results_df.to_csv(
    os.path.join(
        CV_DIR,
        "cv_file_predictions.csv",
    ),
    index=False,
)

cv_window_results_df.to_csv(
    os.path.join(
        CV_DIR,
        "cv_window_predictions.csv",
    ),
    index=False,
)

cv_y_true = cv_window_results_df[
    "true_label"
].to_numpy()

cv_y_pred = cv_window_results_df[
    "pred_label"
].to_numpy()

cv_y_score = cv_window_results_df[
    "reconstruction_mse"
].to_numpy()

overall_cv_metrics = {
    "accuracy": accuracy_score(
        cv_y_true,
        cv_y_pred,
    ),

    "balanced_accuracy":
        balanced_accuracy_score(
            cv_y_true,
            cv_y_pred,
        ),

    "precision": precision_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),

    "recall": recall_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),

    "f1": f1_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),

    "roc_auc": roc_auc_score(
        cv_y_true,
        cv_y_score,
    ),
}

print(
    "\nOVERALL CV WINDOW-LEVEL RESULTS"
)

for name, value in (
    overall_cv_metrics.items()
):
    print(
        f"{name:20s}: "
        f"{value:.6f}"
    )

print(
    classification_report(
        cv_y_true,
        cv_y_pred,
        target_names=[
            "normal",
            "anomaly",
        ],
        zero_division=0,
    )
)

pd.DataFrame([
    overall_cv_metrics
]).to_csv(
    os.path.join(
        CV_DIR,
        "overall_metrics.csv",
    ),
    index=False,
)

cv_confusion_matrix = confusion_matrix(
    cv_y_true,
    cv_y_pred,
    labels=[0, 1],
)

ConfusionMatrixDisplay(
    cv_confusion_matrix,
    display_labels=[
        "normal window",
        "anomaly window",
    ],
).plot(
    values_format="d"
)

plt.title(
    "Combined CV Window-Level Confusion Matrix"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        CV_DIR,
        "confusion_matrix.png",
    ),
    dpi=180,
)

plt.show()



In [ ]:
# ==========================================================
# CELL 14 — train final LSTM model on 100% normal data
# ==========================================================
final_scaler = MinMaxScaler(
    feature_range=(0, 1)
)

final_scaler.fit(
    np.concatenate([
        session["X"]
        for session in healthy_sessions
    ])
)

X_all_healthy = (
    build_windows_from_sessions(
        healthy_sessions,
        final_scaler,
    )
)

print(
    "All pure-normal windows:",
    X_all_healthy.shape,
)

tf.keras.backend.clear_session()

final_model = build_autoencoder_model(
    WINDOW_LEN,
    N_FEATURES,
)

final_history = final_model.fit(
    X_all_healthy,
    X_all_healthy,
    validation_split=0.10,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
    shuffle=True,
)

(
    FINAL_RECON_THRESHOLD,
    FINAL_FEATURE_THRESHOLDS,
    FINAL_MAX_RATIO_THRESHOLD,
) = calibrate_thresholds(
    final_model,
    X_all_healthy,
)

print(
    "\nFINAL THRESHOLDS"
)

print(
    "Final reconstruction threshold:",
    FINAL_RECON_THRESHOLD,
)

print(
    "Final maximum feature-ratio threshold:",
    FINAL_MAX_RATIO_THRESHOLD,
)

print(
    "\nFinal per-feature thresholds:"
)

for name, threshold in zip(
    FEATURE_COLUMNS,
    FINAL_FEATURE_THRESHOLDS,
):
    print(
        f"{name:24s}: "
        f"{threshold:.10f}"
    )

plt.figure(
    figsize=(9, 4)
)

plt.plot(
    final_history.history[
        "loss"
    ],
    label="train loss",
)

plt.plot(
    final_history.history[
        "val_loss"
    ],
    label="validation loss",
)

plt.xlabel("Epoch")
plt.ylabel("MSE")

plt.title(
    "Final LSTM Autoencoder Training"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        FINAL_DIR,
        "final_training_loss.png",
    ),
    dpi=150,
)

plt.show()


In [ ]:
# ==========================================================
# CELL 15 — final model evaluation
# ==========================================================
rng = np.random.default_rng(
    SEED
)

number_normal_evaluation_files = max(
    1,
    round(
        0.20
        * len(healthy_sessions)
    ),
)

normal_evaluation_ids = rng.choice(
    len(healthy_sessions),
    size=number_normal_evaluation_files,
    replace=False,
)

selected_normal_sessions = [
    healthy_sessions[
        int(index)
    ]
    for index in normal_evaluation_ids
]

fault_groups = {}

for session in fault_sessions:
    fault_groups.setdefault(
        session["fault_label"],
        [],
    ).append(
        session
    )

selected_fault_sessions = [
    group[
        int(
            rng.integers(
                len(group)
            )
        )
    ]
    for group in fault_groups.values()
]

final_evaluation_sessions = (
    selected_normal_sessions
    + selected_fault_sessions
)

print(
    "\nFINAL EVALUATION FILES"
)

for session in final_evaluation_sessions:
    state = session.get(
        "state",
        session.get(
            "fault_label",
            "unknown",
        ),
    )

    print(
        f"label={session['binary_label']} | "
        f"state={state} | "
        f"file={session['file']}"
    )

print(
    "\nNormal evaluation files:",
    len(selected_normal_sessions),
)

print(
    "Fault evaluation files:",
    len(selected_fault_sessions),
)

print(
    "Fault classes:",
    sorted(
        fault_groups.keys()
    ),
)

(
    final_eval_file_df,
    final_eval_window_df,
) = aggregate_predictions(
    final_evaluation_sessions,
    final_model,
    final_scaler,
    FINAL_RECON_THRESHOLD,
    FINAL_FEATURE_THRESHOLDS,
    FINAL_MAX_RATIO_THRESHOLD,
)

final_y_true = final_eval_window_df[
    "true_label"
].to_numpy()

final_y_pred = final_eval_window_df[
    "pred_label"
].to_numpy()

final_y_score = final_eval_window_df[
    "reconstruction_mse"
].to_numpy()

final_metrics = {
    "accuracy": accuracy_score(
        final_y_true,
        final_y_pred,
    ),

    "balanced_accuracy":
        balanced_accuracy_score(
            final_y_true,
            final_y_pred,
        ),

    "normal_accuracy": np.mean(
        final_y_pred[
            final_y_true == 0
        ] == 0
    ),

    "anomaly_accuracy": np.mean(
        final_y_pred[
            final_y_true == 1
        ] == 1
    ),

    "precision": precision_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),

    "recall": recall_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),

    "f1": f1_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),

    "roc_auc": roc_auc_score(
        final_y_true,
        final_y_score,
    ),

    "reconstruction_threshold":
        FINAL_RECON_THRESHOLD,

    "max_ratio_threshold":
        FINAL_MAX_RATIO_THRESHOLD,
}

print(
    "\nFINAL WINDOW-LEVEL RESULTS"
)

for name, value in (
    final_metrics.items()
):
    print(
        f"{name:28s}: "
        f"{value:.6f}"
    )

print(
    classification_report(
        final_y_true,
        final_y_pred,
        target_names=[
            "normal",
            "anomaly",
        ],
        zero_division=0,
    )
)

final_eval_file_df.to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "file_predictions.csv",
    ),
    index=False,
)

final_eval_window_df.to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "window_predictions.csv",
    ),
    index=False,
)

pd.DataFrame([
    final_metrics
]).to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "metrics.csv",
    ),
    index=False,
)

final_confusion_matrix = confusion_matrix(
    final_y_true,
    final_y_pred,
    labels=[0, 1],
)

ConfusionMatrixDisplay(
    final_confusion_matrix,
    display_labels=[
        "normal window",
        "anomaly window",
    ],
).plot(
    values_format="d"
)

plt.title(
    "Final LSTM Autoencoder Window-Level Confusion Matrix"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FINAL_EVAL_DIR,
        "confusion_matrix.png",
    ),
    dpi=180,
)

plt.show()

In [ ]:
# ==========================================================
# CELL 16 — save model, scaler, thresholds, and metadata
# ==========================================================
KERAS_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_lstm_autoencoder.keras",
)

SCALER_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_lstm_scaler.pkl",
)

THRESHOLDS_PATH = os.path.join(
    FINAL_DIR,
    "lstm_deployment_thresholds.npz",
)

METADATA_PATH = os.path.join(
    FINAL_DIR,
    "lstm_model_metadata.csv",
)

final_model.save(
    KERAS_PATH
)

joblib.dump(
    final_scaler,
    SCALER_PATH,
)

np.savez(
    THRESHOLDS_PATH,

    recon_threshold=np.float32(
        FINAL_RECON_THRESHOLD
    ),

    feature_thresholds=(
        FINAL_FEATURE_THRESHOLDS.astype(
            np.float32
        )
    ),

    max_ratio_threshold=np.float32(
        FINAL_MAX_RATIO_THRESHOLD
    ),

    file_anomaly_fraction_threshold=np.float32(
        FILE_ANOMALY_FRACTION_THRESHOLD
    ),
)

pd.DataFrame({
    "feature_id": np.arange(
        N_FEATURES
    ),

    "feature_name":
        FEATURE_COLUMNS,

    "scaler_min":
        final_scaler.data_min_,

    "scaler_max":
        final_scaler.data_max_,

    "feature_error_threshold":
        FINAL_FEATURE_THRESHOLDS,
}).to_csv(
    METADATA_PATH,
    index=False,
)

print(
    "Saved:",
    KERAS_PATH,
)

print(
    "Saved:",
    SCALER_PATH,
)

print(
    "Saved:",
    THRESHOLDS_PATH,
)

print(
    "Saved:",
    METADATA_PATH,
)



In [ ]:
# ==========================================================
# CELL 17.2 — export LSTM TFLite models
# ==========================================================
FLOAT32_TFLITE_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_lstm_autoencoder_float32.tflite",
)

DYNAMIC_TFLITE_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_lstm_autoencoder_dynamic.tflite",
)


def convert_lstm_tflite(model, optimize=False):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]

    converter._experimental_lower_tensor_list_ops = False

    if optimize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    return converter.convert()


# Float32 model
float32_tflite = convert_lstm_tflite(
    final_model,
    optimize=False,
)

with open(FLOAT32_TFLITE_PATH, "wb") as file:
    file.write(float32_tflite)


# Dynamic-range model
dynamic_tflite = convert_lstm_tflite(
    final_model,
    optimize=True,
)

with open(DYNAMIC_TFLITE_PATH, "wb") as file:
    file.write(dynamic_tflite)


print("Float32 TFLite:", FLOAT32_TFLITE_PATH)
print("Dynamic TFLite:", DYNAMIC_TFLITE_PATH)

print("Float32 size KB:", round(len(float32_tflite) / 1024, 2))
print("Dynamic size KB:", round(len(dynamic_tflite) / 1024, 2))

In [ ]:
# ==========================================================
# CELL 17.1 — export complete LSTM model as C header
# ==========================================================
H=os.path.join(FINAL_DIR,"lstm_autoencoder_model.h")
L=["encoder_lstm","latent","decoder_lstm"]
W=[final_model.get_layer(x).get_weights() for x in L]
K,R,B=[x[0] for x in W],[x[1] for x in W],[x[2] for x in W]
O,OB=final_model.get_layer("reconstruction_output").layer.get_weights()
U=[x.shape[0] for x in R]

def a(f,n,x):
    x=np.asarray(x,np.float32)
    s=",".join(f"{v:.9g}f" for v in x.ravel())
    f.write(f"static const float {n}[{x.size}]={{{s}}};\n")

with open(H,"w") as f:
    f.write("#ifndef LSTM_AE_H\n#define LSTM_AE_H\n#include <math.h>\n")
    f.write(f"#define NF {N_FEATURES}\n#define WL {WINDOW_LEN}\n#define U1 {U[0]}\n#define U2 {U[1]}\n#define U3 {U[2]}\n")
    f.write(f"#define DIV {SCALE_DIVISOR:.9g}f\n#define RT {FINAL_RECON_THRESHOLD:.9g}f\n#define MRT {FINAL_MAX_RATIO_THRESHOLD:.9g}f\n")
    for n,x in [("MN",final_scaler.data_min_),("MX",final_scaler.data_max_),("FT",FINAL_FEATURE_THRESHOLDS),
                ("K1",K[0]),("R1",R[0]),("B1",B[0]),("K2",K[1]),("R2",R[1]),("B2",B[1]),
                ("K3",K[2]),("R3",R[2]),("B3",B[2]),("KO",O),("BO",OB)]: a(f,n,x)
    f.write(r'''
static inline float sg(float x){return 1.f/(1.f+expf(-x));}
static inline void st(const float*x,int ni,const float*k,const float*r,const float*b,int u,float*h,float*c){
 float g[32]; for(int q=0;q<4*u;q++){float s=b[q];for(int i=0;i<ni;i++)s+=x[i]*k[i*4*u+q];for(int i=0;i<u;i++)s+=h[i]*r[i*4*u+q];g[q]=s;}
 for(int i=0;i<u;i++){float I=sg(g[i]),F=sg(g[u+i]),G=tanhf(g[2*u+i]),O=sg(g[3*u+i]);c[i]=F*c[i]+I*G;h[i]=O*tanhf(c[i]);}}
static inline int lstm_ae(const float raw[WL][NF],float*mse,float*mr){
 float x[WL][NF],z[WL][U1],y[WL][NF],h1[U1]={0},c1[U1]={0},h2[U2]={0},c2[U2]={0},h3[U3]={0},c3[U3]={0},e[NF]={0};
 for(int t=0;t<WL;t++)for(int j=0;j<NF;j++){float p=raw[t][j]/DIV,d=MX[j]-MN[j];x[t][j]=d?((p-MN[j])/d):0;}
 for(int t=0;t<WL;t++){st(x[t],NF,K1,R1,B1,U1,h1,c1);for(int i=0;i<U1;i++)z[t][i]=h1[i];}
 for(int t=0;t<WL;t++)st(z[t],U1,K2,R2,B2,U2,h2,c2);
 for(int t=0;t<WL;t++){st(h2,U2,K3,R3,B3,U3,h3,c3);for(int j=0;j<NF;j++){float v=BO[j];for(int i=0;i<U3;i++)v+=h3[i]*KO[i*NF+j];y[t][j]=v;}}
 float s=0;for(int t=0;t<WL;t++)for(int j=0;j<NF;j++){float d=x[t][j]-y[t][j],q=d*d;s+=q;e[j]+=q;}
 *mse=s/(WL*NF);*mr=0;for(int j=0;j<NF;j++){float r=(e[j]/WL)/fmaxf(FT[j],1e-12f);if(r>*mr)*mr=r;}
 return (*mse>RT)||(*mr>MRT);}
#endif
''')
print("Saved:",H)
print("Header KB:",round(os.path.getsize(H)/1024,2))

In [ ]:
# ==========================================================
# CELL 18 — export STM32 parameter header
# ==========================================================
PARAMS_HEADER_PATH = os.path.join(
    FINAL_DIR,
    "lstm_autoencoder_params.h",
)

with open(
    PARAMS_HEADER_PATH,
    "w",
    encoding="utf-8",
) as file:

    file.write(
        "#ifndef LSTM_AUTOENCODER_PARAMS_H\n"
    )

    file.write(
        "#define LSTM_AUTOENCODER_PARAMS_H\n\n"
    )

    file.write(
        f"#define LSTM_AE_N_FEATURES "
        f"{N_FEATURES}\n"
    )

    file.write(
        f"#define LSTM_AE_WINDOW_LEN "
        f"{WINDOW_LEN}\n"
    )

    file.write(
        f"#define LSTM_AE_WINDOW_STEP "
        f"{WINDOW_STEP}\n"
    )

    file.write(
        f"#define LSTM_AE_SCALE_DIVISOR "
        f"{SCALE_DIVISOR:.1f}f\n"
    )

    file.write(
        f"#define LSTM_AE_RECON_THRESHOLD "
        f"{FINAL_RECON_THRESHOLD:.10f}f\n"
    )

    file.write(
        f"#define LSTM_AE_MAX_RATIO_THRESHOLD "
        f"{FINAL_MAX_RATIO_THRESHOLD:.10f}f\n\n"
    )

    file.write(
        "static const float "
        "LSTM_AE_MIN[LSTM_AE_N_FEATURES] = {\n"
    )

    for value in (
        final_scaler.data_min_
    ):
        file.write(
            f"    {float(value):.10f}f,\n"
        )

    file.write(
        "};\n\n"
    )

    file.write(
        "static const float "
        "LSTM_AE_MAX[LSTM_AE_N_FEATURES] = {\n"
    )

    for value in (
        final_scaler.data_max_
    ):
        file.write(
            f"    {float(value):.10f}f,\n"
        )

    file.write(
        "};\n\n"
    )

    file.write(
        "static const float "
        "LSTM_AE_FEATURE_ERROR_THRESHOLD"
        "[LSTM_AE_N_FEATURES] = {\n"
    )

    for value in (
        FINAL_FEATURE_THRESHOLDS
    ):
        file.write(
            f"    {float(value):.10f}f,\n"
        )

    file.write(
        "};\n\n"
    )

    file.write(
        "static inline float "
        "lstm_ae_scale_raw("
        "float raw_value, int feature_id)\n"
    )

    file.write(
        "{\n"
    )

    file.write(
        "    float physical = "
        "raw_value / LSTM_AE_SCALE_DIVISOR;\n"
    )

    file.write(
        "    float denominator = "
        "LSTM_AE_MAX[feature_id] - "
        "LSTM_AE_MIN[feature_id];\n"
    )

    file.write(
        "    if (denominator == 0.0f) "
        "return 0.0f;\n"
    )

    file.write(
        "    return "
        "(physical - LSTM_AE_MIN[feature_id]) "
        "/ denominator;\n"
    )

    file.write(
        "}\n\n"
    )

    file.write(
        "static inline int "
        "lstm_ae_window_is_anomaly("
        "float recon_mse, "
        "float max_feature_ratio)\n"
    )

    file.write(
        "{\n"
    )

    if SCORE_MODE == "mean":
        file.write(
            "    return recon_mse > "
            "LSTM_AE_RECON_THRESHOLD;\n"
        )

    elif SCORE_MODE == "max":
        file.write(
            "    return max_feature_ratio > "
            "LSTM_AE_MAX_RATIO_THRESHOLD;\n"
        )

    else:
        file.write(
            "    return "
            "(recon_mse > "
            "LSTM_AE_RECON_THRESHOLD) ||\n"
        )

        file.write(
            "           "
            "(max_feature_ratio > "
            "LSTM_AE_MAX_RATIO_THRESHOLD);\n"
        )

    file.write(
        "}\n\n"
    )

    file.write(
        "#endif\n"
    )

print(
    "Saved:",
    PARAMS_HEADER_PATH,
)


In [ ]:
# ==========================================================
# CELL 19 — manual test function
# ==========================================================
def manual_lstm_autoencoder_prediction(
    test_file,
):
    model = tf.keras.models.load_model(
        KERAS_PATH,

        custom_objects={
            "window_mse_loss":
                window_mse_loss,
        },
    )

    scaler = joblib.load(
        SCALER_PATH
    )

    thresholds = np.load(
        THRESHOLDS_PATH
    )

    reconstruction_threshold = float(
        thresholds[
            "recon_threshold"
        ]
    )

    feature_thresholds = (
        thresholds[
            "feature_thresholds"
        ].astype(
            np.float32
        )
    )

    max_ratio_threshold = float(
        thresholds[
            "max_ratio_threshold"
        ]
    )

    file_fraction_threshold = float(
        thresholds[
            "file_anomaly_fraction_threshold"
        ]
    )

    test_df = read_one(
        test_file
    )

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(
            dtype=np.float32
        ),
        scaler,
    )

    (
        reconstruction_mse,
        per_feature_error,
        worst_feature_ids,
    ) = anomaly_score(
        model,
        X_test,
    )

    (
        flags,
        maximum_feature_ratio,
    ) = decisions_from_errors(
        reconstruction_mse,
        per_feature_error,
        reconstruction_threshold,
        feature_thresholds,
        max_ratio_threshold,
    )

    anomaly_fraction = float(
        np.mean(flags)
    )

    file_decision = (
        "ANOMALY"
        if anomaly_fraction
        >= file_fraction_threshold
        else "HEALTHY"
    )

    result = pd.DataFrame({
        "window": np.arange(
            len(X_test)
        ),

        "start_row":
            starts,

        "time_sec": (
            starts
            + WINDOW_LEN / 2.0
        ) * ROW_PERIOD_SEC,

        "reconstruction_mse":
            reconstruction_mse,

        "reconstruction_threshold":
            reconstruction_threshold,

        "max_feature_ratio":
            maximum_feature_ratio,

        "max_ratio_threshold":
            max_ratio_threshold,

        "worst_feature": [
            FEATURE_COLUMNS[int(index)]
            for index in worst_feature_ids
        ],

        "decision": np.where(
            flags == 1,
            "anomaly",
            "normal",
        ),

        "is_anomaly": flags,
    })

    stem = Path(
        test_file
    ).stem

    output_csv = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}.csv",
    )

    result.to_csv(
        output_csv,
        index=False,
    )

    print("\nWindow decisions:")
    print(result["decision"].value_counts())

    print("\nMost frequent worst features:")
    print(result["worst_feature"].value_counts().head(8))
    print(
        "\nMANUAL LSTM AUTOENCODER TEST"
    )

    print(
        "=" * 60
    )

    print(
        "File:",
        test_file,
    )

    print(
        "Total windows:",
        len(result),
    )
    print("Normal windows:", int(np.sum(flags == 0)))
    
    print(
        "Anomalous windows:",
        int(
            np.sum(flags)
        ),
    )

    print(
        "Anomaly fraction:",
        anomaly_fraction,
    )

    print(
        "File decision:",
        file_decision,
    )

    print(
        "Reconstruction threshold:",
        reconstruction_threshold,
    )

    print(
        "Saved CSV:",
        output_csv,
    )

    plt.figure(
        figsize=(12, 4)
    )

    plt.plot(
        result[
            "time_sec"
        ],
        result[
            "reconstruction_mse"
        ],
    )

    plt.axhline(
        reconstruction_threshold,
        linestyle="--",
        label="reconstruction threshold",
    )

    plt.xlabel(
        "Time [s]"
    )

    plt.ylabel(
        "Reconstruction MSE"
    )

    plt.title(
        f"LSTM Autoencoder Manual Test — {stem}"
    )

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_mse.png",
        ),
        dpi=150,
    )

    plt.show()

    plt.figure(
        figsize=(12, 4)
    )

    plt.step(
        result[
            "time_sec"
        ],
        result[
            "is_anomaly"
        ],
        where="mid",
    )

    plt.yticks(
        [0, 1],
        [
            "normal",
            "anomaly",
        ],
    )

    plt.xlabel(
        "Time [s]"
    )

    plt.ylabel(
        "Decision"
    )

    plt.title(
        f"LSTM Autoencoder Decisions — {stem}"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_timeline.png",
        ),
        dpi=150,
    )

    plt.show()

    return result

In [ ]:
# ==========================================================
# CELL 20 — select manual test file
# ==========================================================
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()

root.withdraw()

root.attributes(
    "-topmost",
    True,
)

TEST_FILE = filedialog.askopenfilename(
    title="Select motor CSV or Excel file",

    filetypes=[
        (
            "Excel files",
            "*.xlsx",
        ),

        (
            "CSV files",
            "*.csv",
        ),

        (
            "All files",
            "*.*",
        ),
    ],
)

root.destroy()

if TEST_FILE:
    manual_result = (
        manual_lstm_autoencoder_prediction(
            TEST_FILE
        )
    )

    print(
        manual_result.head(
            30
        )
    )

else:
    print(
        "No file selected."
    )